In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("qoute_dataset.csv")
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [3]:
quotes=df['quote']
quotes=quotes.str.lower()
quotes

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."
...,...
3033,the past beats inside me like a second heart.
3034,"damn, claire. warn a guy before you do a face-..."
3035,"can you be a girl for a few seconds?""""i'm alwa..."
3036,that's what fiction is for. it's for getting a...


In [4]:
import re
quotes = quotes.apply(lambda x: re.sub(r'[^\w\s]', '', x))
display(quotes.head())

,quote
0,the world as we have created it is a process o...
1,it is our choices harry that show what we trul...
2,there are only two ways to live your life one ...
3,the person be it gentleman or lady who has not...
4,imperfection is beauty madness is genius and i...


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
vocab_size=8723
token=Tokenizer(
    num_words=vocab_size,
)
token.fit_on_texts(quotes)

In [7]:

word_index = token.word_index
print(len(word_index))
list(word_index.items())[:10]

8723


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [8]:
sequence=token.texts_to_sequences(quotes)

In [9]:
X=[]
y=[]
for seq in sequence:
    for i in range(1,len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

len(X)

85211

In [10]:
maxlen=max(len(i) for i in X)
maxlen

745

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [12]:
X_padded=pad_sequences(X,maxlen=maxlen,padding='pre')
X_padded.shape

(85211, 745)

In [13]:
y=np.array(y)

In [14]:
from tensorflow.keras.utils import to_categorical
y_oneHot=to_categorical(y,num_classes=vocab_size)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,SimpleRNN

In [16]:
embedding_dim=50
rnn_units=128

In [17]:
model=Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=maxlen),
])
model.add(
    SimpleRNN(units=rnn_units)
)
model.add(
    Dense(units=vocab_size,activation='softmax')
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.compile(
    optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy']
    )


In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
lstm_model=Sequential([
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=maxlen),
])
lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(Dense(units=vocab_size,activation='softmax'))
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
epochs=100
batch_size=128

In [22]:
lstm_model.compile(
    optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']
)
history = lstm_model.fit(
    X_padded, y_oneHot, epochs=epochs, batch_size=batch_size, verbose=1,validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.0389 - loss: 6.7089 - val_accuracy: 0.0456 - val_loss: 6.6364
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0580 - loss: 6.2923 - val_accuracy: 0.0651 - val_loss: 6.5223
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 43s 57ms/step - accuracy: 0.0805 - loss: 6.0285 - val_accuracy: 0.0886 - val_loss: 6.4193
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.0999 - loss: 5.8037 - val_accuracy: 0.0973 - val_loss: 6.3842
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1120 - loss: 5.6150 - val_accuracy: 0.1057 - val_loss: 6.3571
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1209 - loss: 5.4464 - val_accuracy: 0.1078 - val_loss: 6.3745
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 56ms/step - accuracy: 0.1293 - loss: 5.2891 - val_accuracy: 0.1114 - val_loss: 6.3936
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1362 - loss: 5

In [24]:
lstm_model.save("lstm_model.h5")

In [25]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [36]:
def predictor(model,token,text,maxlen):
  text = text.lower()

  seq = token.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=maxlen, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [39]:
seed_text = "source of happiness is "
next_word = predictor(lstm_model,token,seed_text,maxlen)
print(next_word)

yourself


In [40]:
def generate_text(model,token,seed_text,maxlen,n_words):
  for _ in range(n_words):
    next_word = predictor(model,token,seed_text,maxlen)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [42]:
seed_text="Reason behind success  "
generate_text(lstm_model,token,seed_text,maxlen,10)

'Reason behind success   were never too late it wasnt but im not saying'

In [43]:
import pickle


In [44]:
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(token, f)

In [45]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(maxlen, f)